# Defect Formation Energy of a Boron Vacancy in h-BN

> **Fabian Bertoldo, Sajid Ali, Simone Manti & Kristian S. Thygesen**,
> "Quantum point defects in 2D materials - the QPOD database", Nature, 2022.
> [DOI:10.1038/s41524-022-00730-w](https://doi.org/10.1038/s41524-022-00730-w)

Computes the neutral formation energy of the vacancy created in the
[structure notebook](defect_point_vacancy_boron_nitride.ipynb) and compares it with QPOD's
`v_B in BN (charge 0)` entry, using the generic
[Defect Formation Energy](../workflows/defect_formation_energy.ipynb) workflow.

$$E_f = E_{\text{defective}} - E_{\text{pristine}} - \Delta N_B\, \mu_B \quad [\text{eV}]$$

$\mu_B$ is the total energy of boron's standard state per atom (QPOD's convention, $q = 0$).

QPOD's value is for a relaxed 84-atom cell (15.06 Å defect spacing); this notebook's 48-atom
cell is unrelaxed -- both effects together shift E_f by about 0.06 eV (measured with MACE).


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Names saved by defect_point_vacancy_boron_nitride.ipynb.
PRISTINE_NAME = "h-BN supercell"
DEFECTIVE_NAME = "B-vacancy h-BN"


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
DEFECT_WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
MY_WORKFLOW_NAME = "Defect Formation Energy"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "cluster-001"
QUEUE_NAME = QueueName.OF
PPN = 40
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds


### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "paw"  # B, N: paw or nc only -- no ultrasoft PBE pseudopotential exists
ECUTWFC = 40  # higher of B/N PseudoDojo "high" hints (15, 20 Ha), in Ry
ECUTRHO = 8 * ECUTWFC  # PAW needs a much denser augmentation charge

KPOINT_DENSITY = 6  # paper's ground-state k-point density (Bertoldo et al. 2022, 6.1), Å⁻¹


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load from the uploads folder, and print provenance

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.material import load_material_from_folder

def formula(material):
    counts = Counter(material.basis.elements.values)
    return "".join(f"{element}{counts[element]}" for element in sorted(counts))

materials_by_name = {}
for name in (PRISTINE_NAME, DEFECTIVE_NAME):
    material = load_material_from_folder(FOLDER, name, verbose=False)
    if material is None:
        raise ValueError(f"No material named '{name}' in '{FOLDER}'. Run "
                         "defect_point_vacancy_boron_nitride.ipynb first, or correct the name above.")
    materials_by_name[name] = material

pristine, defective = materials_by_name[PRISTINE_NAME], materials_by_name[DEFECTIVE_NAME]
for name, material in materials_by_name.items():
    a, b = material.lattice.a, material.lattice.b
    print(f"{name}: {formula(material)}, {len(material.basis.elements.ids)} atoms, "
          f"cell {a:.2f} x {b:.2f} Å, defect-defect distance {min(a, b):.2f} Å")


### 3.2. Resolve elemental reference materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

elements = sorted(set(pristine.basis.elements.values) | set(defective.basis.elements.values))

# Materials.get_by_categories("elemental") returns nothing -- "elemental" is a per-entry tag,
# not a registered category.
elemental_by_symbol = {}
for entry in Materials.get_as_list():
    symbols = {atom["value"] for atom in entry["basis"]["elements"]}
    if len(symbols) == 1 and symbols <= set(elements):
        elemental_by_symbol[symbols.pop()] = entry

missing = set(elements) - elemental_by_symbol.keys()
if missing:
    raise ValueError(f"No Standata elemental reference material for {missing}.")

elemental_materials = {element: Material.create(elemental_by_symbol[element]) for element in elements}
for element, material in elemental_materials.items():
    print(f"{element}: {material.name}")


### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pristine = Material.create(get_or_create_material(client, pristine, ACCOUNT_ID))
saved_defective = Material.create(get_or_create_material(client, defective, ACCOUNT_ID))
saved_elemental = {
    element: Material.create(get_or_create_material(client, material, ACCOUNT_ID))
    for element, material in elemental_materials.items()
}


## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
print(f"Using application: {app.name}, model: {model_config['subtype']}/{model_config['functional']}, "
      f"pseudopotential: {PSEUDOPOTENTIAL_TYPE}")


### 4.1. k-grid per material

In [ ]:
import math

def kgrid_for_density(material, periodic_dims=(0, 1, 2)):
    # |b_i| = 2*pi*reciprocal_vector_norms[i]; dims outside periodic_dims stay at 1 (vacuum).
    norms = material.lattice.reciprocal_vector_norms
    grid = [1, 1, 1]
    for dim in periodic_dims:
        grid[dim] = max(1, math.ceil(KPOINT_DENSITY * 2 * math.pi * norms[dim]))
    return grid


kgrid = {
    PRISTINE_NAME: kgrid_for_density(pristine, periodic_dims=(0, 1)),
    DEFECTIVE_NAME: kgrid_for_density(defective, periodic_dims=(0, 1)),
    **{element: kgrid_for_density(material) for element, material in elemental_materials.items()},
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")


## 5. Configure compute
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), clusters[0])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Prerequisite Total Energy jobs

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.core.entity.property.api import find_total_energy_for_material
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid
from mat3ra.notebooks_utils.job import create_job

total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)

# Reuses a finished Total Energy job per material if one exists; creates one otherwise.
prerequisite_materials = {PRISTINE_NAME: saved_pristine, **saved_elemental}
prerequisite_job_ids = []
for name, saved_material in prerequisite_materials.items():
    if find_total_energy_for_material(client, saved_material.id, source="my_account") is not None:
        print(f"♻️  {name}: reusing existing Total Energy")
        continue
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = f"Total Energy {name}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model
    unit = subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(cutoffs_context)
    subworkflow.set_unit(unit)
    apply_scf_kgrid(workflow, kgrid[name], material=saved_material)

    job_response = create_job(
        api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, prefix=f"{workflow.name} {timestamp}", compute=compute.to_dict(),
    )
    prerequisite_job_ids.append(job_response["_id"])
    print(f"✅ {name}: created Total Energy job {job_response['_id']}")


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if prerequisite_job_ids:
    submit_jobs(client.jobs, prerequisite_job_ids)
    print(f"✅ Submitted {len(prerequisite_job_ids)} prerequisite job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, prerequisite_job_ids, poll_interval=POLL_INTERVAL)


## 7. Configure the Defect Formation Energy workflow

In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    DEFECT_WORKFLOW_SEARCH_TERM
)
defect_workflow = Workflow.create(defect_workflow_config)
defect_workflow.name = MY_WORKFLOW_NAME

for subworkflow in defect_workflow.subworkflows:
    if subworkflow.name == "Compute Total Energy for Defective Material":
        subworkflow.model = model
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        for context in (cutoffs_context,
                        PointsGridDataProvider(material=defective, dimensions=kgrid[DEFECTIVE_NAME],
                                               isEdited=True).get_context_item_data()):
            unit.add_context(context)
        subworkflow.set_unit(unit)
    elif subworkflow.name == "Resolve Total Energies for Elemental Materials":
        # References resolve from this account (falling back to curators), not the workflow's
        # own default 'public' (any owner, highest precision wins).
        source_unit = subworkflow.get_unit_by_name(name="assign-source-of-te-for-an-element")
        source_unit.value = "'my_account'"
        subworkflow.set_unit(source_unit)

# nspin/magnetization on the defective cell's SCF only; the references are non-magnetic.
patch_workflow_qe_input(defect_workflow, {"system": {"nspin": 2, "starting_magnetization(1)": 0.5}},
                        unit_names=["pw_scf"])

visualize_workflow(defect_workflow)


## 8. Create and run the Defect Formation Energy job
### 8.1. Create the job (defective + pristine)

In [ ]:
defect_job_response = create_job(
    api_client=client, materials=[saved_defective, saved_pristine], workflow=defect_workflow,
    project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
    prefix=f"{MY_WORKFLOW_NAME} {saved_defective.formula} {timestamp}",
)
defect_job_id = defect_job_response["_id"]
print(f"✅ Defect Formation Energy job created: {defect_job_id}")


### 8.2. Submit and monitor the job

In [ ]:
if defect_job_id:
    client.jobs.submit(defect_job_id)
    print(f"✅ Job {defect_job_id} submitted successfully!")
    await wait_for_jobs_to_finish_async(client.jobs, [defect_job_id], poll_interval=POLL_INTERVAL)


## 9. Retrieve the results
### 9.1. Defect formation energy

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

defect_energy_data = get_properties_for_job(client, defect_job_id, property_name="defect_formation_energy")
if not defect_energy_data:
    raise RuntimeError(f"Job {defect_job_id} produced no defect_formation_energy -- check it finished.")
visualize_properties(defect_energy_data, title="Defect Formation Energy")
e_formation = defect_energy_data[0]["value"]


### 9.2. Compare with QPOD

Only the neutral (q = 0) defect is compared: QPOD's charged states need a finite-size correction
this workflow does not compute.

In [ ]:
QPOD = {"standard_states": 10.18, "B_poor": 8.89}  # eV, QPOD v_B in BN (charge 0)

difference = e_formation - QPOD["standard_states"]
print(f"E_f (this notebook):        {e_formation:.3f} eV")
print(f"E_f (QPOD, standard states): {QPOD['standard_states']:.3f} eV")
print(f"E_f (QPOD, B-poor):          {QPOD['B_poor']:.3f} eV")
print(f"Difference from standard states: {difference:+.3f} eV")
print(f"Reproduces Bertoldo et al. (2022): {'yes' if abs(difference) <= 0.5 else 'no'}")
